INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA DIRECTA (SIN CONTINGENTES) (CNBS)


In [2]:
import requests
import pandas as pd
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

urlIndicadores= f"https://datos.cnbs.gob.hn/es/api/3/action/datastore_search" 
logo=["FICOHSA","BANCATLAN","BAC CREDOMATIC","BANCOCCI"]
#logo=["FICOHSA"]

for logos in logo:
        params1={
                "resource_id":"509e19c4-09d1-4f3d-9ec4-f7a6e874bb78",  # Recurso de Indicadores
                "filters":{
                        "Logo":logo,
                        "Indicador":"INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA DIRECTA (SIN CONTINGENTES)",
                        "FechaReporte": "2025-06-30"
                },
                "limit":1000      
        }
    
    
        responseIndicadores = requests.get(urlIndicadores, json=params1,verify=False)
            
    
        if responseIndicadores.status_code == 200:
                datosIndicadores = responseIndicadores.json()["result"]["records"]
                df_Indicadores=pd.DataFrame(datosIndicadores)
        else:
                print(f"Error {responseIndicadores.status_code}: {responseIndicadores.text}")
else:
    print(f"Error {responseIndicadores.status_code}: {responseIndicadores.text}")     

df_final=df_Indicadores[['FechaReporte','Logo','Indicador','TipoIndicador','Saldo']]
df_final=df_final.rename(columns={
    'FechaReporte':'FECHA',
    'Logo':'BANCO',
    'Indicador':'NOMBRE_INDICADOR',
    'TipoIndicador':'TIPO',
    'Saldo':'VALOR'
})

df_final.sort_values(by='FECHA', ascending=True,inplace=True)

Error 200: {"help": "https://datos.cnbs.gob.hn/es/api/3/action/help_show?name=datastore_search", "success": true, "result": {"filters": {"Logo": ["FICOHSA", "BANCATLAN", "BAC CREDOMATIC", "BANCOCCI"], "Indicador": "INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA DIRECTA (SIN CONTINGENTES)", "FechaReporte": "2025-06-30"}, "include_total": true, "limit": 1000, "records_format": "objects", "resource_id": "509e19c4-09d1-4f3d-9ec4-f7a6e874bb78", "total_estimation_threshold": null, "records": [{"_id":8311,"Tipo":"01","Inst":"01","TipoInstitucion":"BANCOS COMERCIALES","Logo":"BANCATLAN","FechaReporte":"2025-06-30","Indicador":"INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA DIRECTA (SIN CONTINGENTES)","TipoIndicador":"2) INDICADORES DE CALIDAD DE ACTIVOS","Saldo":2.20655089929103,"Linea":39,"Publicacion":"Si","TipoIndicador_ID":2},{"_id":8313,"Tipo":"01","Inst":"03","TipoInstitucion":"BANCOS COMERCIALES","Logo":"BANCOCCI","FechaReporte":"2025-06-30","Indicador":"INDICE DE MOROSIDAD SOBRE CARTERA CR

In [3]:
df_final.head(10)


,FECHA,BANCO,NOMBRE_INDICADOR,TIPO,VALOR
0,2025-06-30,BANCATLAN,INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA D...,2) INDICADORES DE CALIDAD DE ACTIVOS,2.206551
1,2025-06-30,BANCOCCI,INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA D...,2) INDICADORES DE CALIDAD DE ACTIVOS,5.065812
2,2025-06-30,FICOHSA,INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA D...,2) INDICADORES DE CALIDAD DE ACTIVOS,2.604437
3,2025-06-30,BAC CREDOMATIC,INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA D...,2) INDICADORES DE CALIDAD DE ACTIVOS,1.645352


In [4]:
df_resultado=df_final.drop_duplicates()
df_resultado.head(10)

,FECHA,BANCO,NOMBRE_INDICADOR,TIPO,VALOR
0,2025-06-30,BANCATLAN,INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA D...,2) INDICADORES DE CALIDAD DE ACTIVOS,2.206551
1,2025-06-30,BANCOCCI,INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA D...,2) INDICADORES DE CALIDAD DE ACTIVOS,5.065812
2,2025-06-30,FICOHSA,INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA D...,2) INDICADORES DE CALIDAD DE ACTIVOS,2.604437
3,2025-06-30,BAC CREDOMATIC,INDICE DE MOROSIDAD SOBRE CARTERA CREDITICIA D...,2) INDICADORES DE CALIDAD DE ACTIVOS,1.645352


In [5]:
################           INSERT TABLE / DATA           #################   #      
from azure.identity import InteractiveBrowserCredential
import pandas as pd
from Server import AZURE
from tqdm import tqdm
from sqlalchemy import create_engine, text

credential = InteractiveBrowserCredential()

server = AZURE
database = 'sqlpooldwhandr01'
schema = 'HN_NAP_HO_MISRIESGOS_F'
tabla = 'CNBS_INDICADORES_FINANCIEROS'
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
        f"DRIVER={driver};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
        f"Authentication=ActiveDirectoryInteractive;"
    )

connection_uri = f"mssql+pyodbc:///?odbc_connect={connection_string}"
engine = create_engine(connection_uri, fast_executemany=True)


query = text("""
SELECT COLUMN_NAME, DATA_TYPE , CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = :tablita
AND TABLE_SCHEMA = :esquema
""")

with engine.connect() as conn:
    result = conn.execute(query, {"tablita": tabla, "esquema":schema})
    columns_types = {row[0]: [row[1] , row[2]] for row in result}
#columns_types

In [6]:
#-----------------------------############### INSERT ##################------------------------------------#
#tmp_df = df_cleaned.iloc[:100].copy()


#data_frame = pd.DataFrame(js)
df_resultado['FECHA']=pd.to_datetime(df_resultado['FECHA']).dt.strftime('%Y-%m-%d')
data_frame=df_resultado
chunksize = 100

for start in tqdm(range(0, len(data_frame), chunksize), desc="Insertando datos"):
    end = min(start + chunksize, len(data_frame))
    chunk = data_frame.iloc[start:end]

    try:
        chunk.to_sql(tabla, 
                     con=engine, 
                     schema=schema, 
                     if_exists='append', 
                     index=False, 
                     chunksize=chunksize)
    except Exception as e:
        print(f"Error al insertar datos en la base de datos: {e}")
    #break
print('Proceso de insercion completado')

Insertando datos:   0%|          | 0/1 [00:00<?, ?it/s]

Insertando datos: 100%|██████████| 1/1 [00:01<00:00,  1.87s/it]

Proceso de insercion completado
